# 思考如何修改 Thought-Action-Observation 循环来实现这些功能

## 功能一：添加"记忆"功能，记住用户偏好

**思路：** 在循环中增加一个记忆存储模块，不依赖外部数据库。

```python
# 初始化一个记忆字典，放在循环外面
user_memory = {}  # {"偏好类型": "值", ...}

# 在每轮循环的 Observation 之后，加一步：提取偏好
# 在 available_tools 中新增一个工具
def remember_user_preference(key: str, value: str) -> str:
    user_memory[key] = value
    return f"已记住用户的偏好：{key} = {value}"

# 在系统 prompt 中增加指令：
# "当用户在对话中表达偏好时（如'我喜欢历史景点'、'预算在500以内'），
#  调用 remember_user_preference 存储。在后续推荐时，优先考虑已存储的偏好。"
```

**在循环中的位置：** Thought → Action（如果用户表达了偏好，Action 中调用 `remember_user_preference`）→ Observation（存储结果）→ 下一轮 Thought 时 LLM 会读到 Observation 中的偏好记录。

## 功能二：景点门票售罄时自动推荐备选

**思路：** 增加一个门票查询工具，工具返回"售罄"时自动触发重试。

```python
def check_ticket(scenic_name: str) -> str:
    # 模拟门票 API
    tickets = {"颐和园": "有票", "故宫": "售罄", "长城": "有票"}
    return tickets.get(scenic_name, "未知景点")

# 在 system prompt 中增加规则：
# "调用 get_attraction 获得推荐后，必须调用 check_ticket 检查门票。
#  如果返回'售罄'，自动调用 get_attraction 搜索下一个备选景点。
#  最多尝试 3 个备选，之后向用户说明情况。"
```

**关键修改：** 不是简单在 prompt 里说一句，而是在代码里给 `Observation` 做增强——当 `check_ticket` 返回"售罄"时，`Observation` 中附加 `/需要重新搜索备选方案` 指令，促使 LLM 在下一轮 Thought 中重新调用 `get_attraction`。

## 功能三：用户连续拒绝 3 次后反思调整策略

**思路：** 在循环外维护拒绝计数器，拒绝达到阈值时注入反思指令。

```python
reject_count = 0  # 循环外面

# 在每次循环的 Observation 后检查
if "用户拒绝了" in observation or "不想去" in observation:
    reject_count += 1
else:
    reject_count = 0  # 用户接受了某个推荐，重置

if reject_count >= 3:
    # 在 Observation 末尾追加反思指令
    observation += "\n[系统提示] 用户已连续拒绝 3 个推荐。请反思：之前的推荐方向是否与用户偏好不符？调整推荐策略，尝试完全不同类型的景点，或直接询问用户更具体的需求。"

if reject_count >= 5:
    print("用户连续拒绝过多，终止循环，请求人工介入。")
    break
```

**为什么不能只靠 prompt？** 光在 prompt 里说"连续拒绝就反思"，LLM 可能不会自己数到底拒绝了几次——它的注意力在长上下文中会衰减。**在代码层面维护计数器**，达到阈值时主动在 Observation 中注入提醒，才能确保 LLM 真正"意识到"该反思了。